<a href="https://colab.research.google.com/github/madelsu/Dynamic_Hyperparameter_Optimization_LLM/blob/main/(a)SQL_Database/Adding_narratives.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Excel (caseid + narrative) → TXT (tab-delimited) → CHUNKED load to MySQL → UPDATE main.narrative

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.dialects.mysql import LONGTEXT, VARCHAR

# ======================
# SETTINGS — EDIT THESE
# ======================
DATA_DIR     = r"C:\Projects\Manuela Del Castillo\Data"
EXCEL_PATH   = rf"{DATA_DIR}\cases.xlsx"          # your narratives Excel
SHEET_NAME   = 0                            # or 0 for first sheet
TXT_OUT      = rf"{DATA_DIR}\ICSR_narratives_clean.txt"     # tab-delimited output

DB_URL       = "mysql+mysqlconnector://root:@..../db_icsr_assessment_manuela"
MAIN_TABLE   = "icsr_assessment_import"
MAIN_CASE_ID_COL = "case_id"                         # <-- set to your real join key in MAIN_TABLE
MAIN_NARR_COL = "case_narrative"   # <-- your main table's narrative column

# Column names in the Excel file:
SRC_CASEID_COL    = "caseid"
SRC_NARRATIVE_COL = "narrative"

# Behavior
CHUNK_SIZE          = 20_000    # TXT → DB insert chunk size
OVERWRITE_EXISTING  = False     # False: fill blanks only; True: overwrite existing narrative


In [ ]:
# =========================
# A) Excel → TXT (tab-delim)
# =========================
print("🔄 Reading Excel and writing TSV...")
df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME, dtype=str)

# normalize headers
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'[^0-9a-zA-Z_]+', '_', regex=True)
      .str.replace(r'__+', '_', regex=True)
      .str.strip('_')
)

# keep required cols
missing = {SRC_CASEID_COL, SRC_NARRATIVE_COL} - set(df.columns)
if missing:
    raise ValueError(f"Excel missing columns: {missing}. Found: {list(df.columns)}")

df = df[[SRC_CASEID_COL, SRC_NARRATIVE_COL]].copy()

# clean values (keep one line per row in TSV)
def _clean(s):
    if pd.isna(s): return ""
    s = str(s).replace("\t", " ").replace("\r\n", " ").replace("\n", " ").replace("\r", " ")
    return s.strip()

df[SRC_CASEID_COL]    = df[SRC_CASEID_COL].map(_clean)
df[SRC_NARRATIVE_COL] = df[SRC_NARRATIVE_COL].map(_clean)

# remove empties and dedup by caseid (keep last)
df = df[df[SRC_NARRATIVE_COL].astype(str).str.len() > 0]
df = df.drop_duplicates(subset=[SRC_CASEID_COL], keep="last")

# write TSV (folder already exists)
df.to_csv(TXT_OUT, sep="\t", index=False)
print(f"✅ Saved TXT at: {TXT_OUT} ({len(df)} rows)")

In [ ]:
# =========================
# C) CODE FOR ALL CASES IN BATCHES
# =========================
# Fill ALL empty narratives using digits-only matching on BOTH sides

from sqlalchemy import create_engine, text

# ---- EDIT if needed ----
DB_URL = "mysql+mysqlconnector://root:@.../db_icsr_assessment_manuela?charset=utf8mb4"

MAIN_DB, MAIN_TABLE   = "db_icsr_assessment_manuela", "icsr_assessment_import"
MAIN_CASE_ID_COL      = "case_id"          # can contain non-digits; we'll extract digits
MAIN_NARR_COL         = "case_narrative"

STAGING_DB, STAGING_T = "icsr_staging", "narratives"  # caseid (VARCHAR), narrative (LONGTEXT)

BATCH_SIZE = 1000
DRY_RUN    = False    # True = show counts only (no updates)

engine = create_engine(DB_URL, pool_pre_ping=True)

# --- 1) Build a mapping: digits_id -> "best" narrative from staging ---
# "best" = longest non-empty narrative for that ID, using digits extracted from either
#          the narrative text or (fallback) the caseid.
print("📦 Building digits→narrative map from staging…")
with engine.connect() as conn:
    rows = conn.execute(text(f"""
        WITH stg AS (
            SELECT
                -- prefer digits inside narrative; else digits from caseid
                COALESCE(
                    REGEXP_SUBSTR(narrative, '[0-9]{{6,}}'),
                    REGEXP_REPLACE(caseid, '[^0-9]', '')
                )                         AS key_digits,
                narrative,
                LENGTH(narrative)         AS len
            FROM {STAGING_DB}.{STAGING_T}
            WHERE narrative IS NOT NULL AND TRIM(narrative) <> ''
        ),
        ranked AS (
            SELECT
                key_digits, narrative, len,
                ROW_NUMBER() OVER (
                    PARTITION BY key_digits
                    ORDER BY len DESC
                ) AS rn
            FROM stg
            WHERE key_digits IS NOT NULL AND key_digits <> ''
        )
        SELECT key_digits, narrative
        FROM ranked
        WHERE rn = 1;
    """)).fetchall()

staging_map = {r._mapping["key_digits"]: r._mapping["narrative"] for r in rows}
print(f"✅ Staging map size: {len(staging_map):,} unique IDs")

# --- 2) Pull main rows that still need a narrative (with their digits) ---
print("🔎 Loading main IDs with empty/NULL narrative…")
with engine.connect() as conn:
    main_rows = conn.execute(text(f"""
        SELECT
          {MAIN_CASE_ID_COL} AS case_id_raw,
          REGEXP_REPLACE(CAST({MAIN_CASE_ID_COL} AS CHAR), '[^0-9]', '') AS case_id_digits
        FROM {MAIN_DB}.{MAIN_TABLE}
        WHERE {MAIN_NARR_COL} IS NULL OR TRIM({MAIN_NARR_COL}) = ''
        ORDER BY {MAIN_CASE_ID_COL};
    """)).fetchall()

to_update = []
for r in main_rows:
    digits = r._mapping["case_id_digits"]
    if digits and digits in staging_map:
        to_update.append({
            "d": digits,
            "nar": staging_map[digits],
        })

print(f"🧮 Candidates to update (matched by digits on both sides): {len(to_update):,}")

In [ ]:
if DRY_RUN:
    print("DRY_RUN = True → no updates performed.")
else:
    # --- 3) Update in batches; print progress ---
    total = 0
    for i in range(0, len(to_update), BATCH_SIZE):
        batch = to_update[i:i+BATCH_SIZE]
        with engine.begin() as conn:
            res = conn.execute(text(f"""
                UPDATE {MAIN_DB}.{MAIN_TABLE}
                SET {MAIN_NARR_COL} = :nar
                WHERE ({MAIN_NARR_COL} IS NULL OR TRIM({MAIN_NARR_COL}) = '')
                  AND REGEXP_REPLACE(CAST({MAIN_CASE_ID_COL} AS CHAR), '[^0-9]', '') = :d;
            """), batch)  # executemany
        total += res.rowcount or 0
        print(f"🔹 Batch {i//BATCH_SIZE + 1}: updated {res.rowcount or 0} (total {total})")

    print(f"🎉 Done. Total rows updated: {total:,}")
